# Metacognitive Medical Digital Twins Pipeline

This notebook implements the complete MDT pipeline with MIMIC-IV and Medical-O1 data sources.

**Alignment Components:**
- Theory of Mind module for user belief inference
- Composite reward engine (5 components: safety, empathy, proactivity, metacognition, semantic)
- GRPO training for multi-objective alignment

**Ontology Components:**
- LOINC, SNOMED-CT, ICD-10 code mappings
- Clinical reference ranges validation
- MIMIC-IV item ID mappings

**Data Sources:**
- MIMIC-IV (ICU trajectories)
- Medical-O1 (reasoning chains)


## 1. Setup & Environment

In [3]:
# Environment setup
import sys
sys.path.append(".")

# Core imports
from config.configs import DataConfig
from data.mimic_processor import MIMICProcessor
from data.medical_o1_processor import MedicalO1Processor
from core.theory_of_mind import TheoryOfMindModule
from rewards.composite_engine import CompositeRewardEngine
from training.grpo_trainer import run_grpo_training
from utils.ontology_validator import OntologyValidator
from utils.helpers import clean_memory, setup_logging

print("✓ All components imported successfully")


✓ All components imported successfully


## 2. Configuration

In [4]:
# Setup logging
setup_logging()
import logging
logger = logging.getLogger(__name__)

logger.info("="*80)
logger.info("MEDICAL DIGITAL TWIN - MASTER PIPELINE")
logger.info("="*80)

# Load configuration
data_config = DataConfig()
print(f"✓ Configuration loaded")
print(f"  MIMIC patients: {data_config.max_patients}")
print(f"  Medical-O1 examples: {data_config.max_o1_examples}")


2026-03-26 11:09:19,324 - __main__ - INFO - ================================================================================
2026-03-26 11:09:19,325 - __main__ - INFO - MEDICAL DIGITAL TWIN - MASTER PIPELINE
2026-03-26 11:09:19,325 - __main__ - INFO - ================================================================================
✓ Configuration loaded
  MIMIC patients: 1000
  Medical-O1 examples: 5000


## 3. Data Loading

In [5]:
# Load MIMIC-IV data
print("Loading MIMIC-IV data...")
mimic_processor = MIMICProcessor(data_config)

if mimic_processor.check_availability():
    mimic_data = mimic_processor.process_all_patients(
        max_patients=data_config.max_patients
    )
    print(f"✓ Loaded {len(mimic_data)} MIMIC examples")
    
    # Load ICD diagnoses for validation
    try:
        icd_diagnoses = mimic_processor.load_icd_diagnoses()
        print(f"✓ Loaded {len(icd_diagnoses)} ICD diagnoses")
    except Exception as e:
        print(f"⚠️ ICD diagnoses loading failed: {e}")
        icd_diagnoses = None
else:
    print("⚠️ MIMIC data not available")
    mimic_data = []
    icd_diagnoses = None

Loading MIMIC-IV data...
2026-03-26 11:09:27,751 - data.mimic_processor - INFO - Initialized MIMIC-IV v3.1 processor
2026-03-26 11:09:27,752 - data.mimic_processor - INFO - Root directory: mimiciv/3.1
2026-03-26 11:09:27,752 - data.mimic_processor - INFO - Loaded 61 LOINC codes from ontology
2026-03-26 11:09:27,753 - data.mimic_processor - INFO - Loaded 19 chartevents item categories
2026-03-26 11:09:27,753 - data.mimic_processor - INFO - Loaded 32 reference ranges
2026-03-26 11:09:27,758 - data.mimic_processor - INFO - All required MIMIC-IV files found
2026-03-26 11:09:27,759 - data.mimic_processor - INFO - Processing up to 1000 patients...
2026-03-26 11:09:27,759 - data.mimic_processor - INFO - Loading ICU stays from: mimiciv/3.1/icu/icustays.csv
2026-03-26 11:09:27,904 - data.mimic_processor - INFO - Loaded 94458 ICU stays
2026-03-26 11:09:27,910 - data.mimic_processor - INFO - Selected 1000 ICU stays
2026-03-26 11:09:27,910 - data.mimic_processor - INFO - Extracting vital signs for

In [6]:
# Load Medical-O1 data
print("Loading Medical-O1 data...")
o1_processor = MedicalO1Processor()

try:
    o1_dataset = o1_processor.load_dataset(
        split='train',
        config=data_config.medical_o1_config
    )
    if o1_dataset:
        o1_data = o1_processor.format_for_training(
            o1_dataset,
            max_examples=data_config.max_o1_examples
        )
        print(f"✓ Loaded {len(o1_data)} Medical-O1 examples")
    else:
        print("⚠️ Medical-O1 dataset loading failed")
        o1_data = []
except Exception as e:
    print(f"⚠️ Medical-O1 loading failed: {e}")
    o1_data = []

# Combine datasets
all_training_data = mimic_data + o1_data
print(f"✓ Total training examples: {len(all_training_data)}")
print(f"  MIMIC-IV: {len(mimic_data)}")
print(f"  Medical-O1: {len(o1_data)}")

Loading Medical-O1 data...
2026-03-26 11:09:33,917 - data.medical_o1_processor - INFO - Initialized MedicalO1Processor
2026-03-26 11:09:33,918 - data.medical_o1_processor - INFO - Loading FreedomIntelligence/medical-o1-reasoning-SFT (config=en) from HuggingFace...
2026-03-26 11:09:33,992 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:09:34,005 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/README.md "HTTP/1.1 200 OK"
2026-03-26 11:09:34,047 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/resolve/fc2c9e8a37b38f38da6d449564a8c350b244aef4/medical-o1-reasoning-SFT.py "HTTP/1.1 404 Not Found"
2026-03-26 11:09:34,165 - httpx - INFO - HTTP Request: HEAD https://s3.amazona

## 3.5. Ontology Validation

## 4. Model Training (SFT)

In [8]:
# Initialize ontology validator
print("Initializing ontology validator...")
ontology_validator = OntologyValidator()

# Validate ICD codes if available
if icd_diagnoses is not None:
    print("Validating ICD-10 codes...")
    valid_icd_count = 0
    total_icd_count = len(icd_diagnoses)
    
    # Sample validation (check first 100 for speed)
    sample_icd = icd_diagnoses.head(100)
    for _, row in sample_icd.iterrows():
        icd_code = row['icd_code']
        if ontology_validator.validate_icd10_code(icd_code):
            valid_icd_count += 1
    
    print(f"✓ ICD-10 validation: {valid_icd_count}/{len(sample_icd)} sample codes valid")
    print(f"  Total ICD diagnoses available: {total_icd_count}")
else:
    print("⚠️ ICD diagnoses not available for validation")

# Validate LOINC codes from loaded data
print("Validating LOINC codes...")
loinc_codes = ontology_validator.loinc_codes
print(f"✓ Loaded {len(loinc_codes)} LOINC codes in ontology")

# Test some common LOINC codes
test_loinc = ['8867-4', '8480-6', '8462-4', '2524-7', '2160-0']  # HR, SBP, DBP, Lactate, Creatinine
valid_loinc_count = 0
for code in test_loinc:
    if ontology_validator.validate_loinc_code(code):
        valid_loinc_count += 1

print(f"✓ LOINC validation: {valid_loinc_count}/{len(test_loinc)} test codes valid")

# Validate SNOMED-CT codes
print("Validating SNOMED-CT codes...")
snomed_codes = ontology_validator.snomed_codes
print(f"✓ Loaded {len(snomed_codes)} SNOMED-CT codes in ontology")

# Test some common SNOMED codes
test_snomed = ['91302008', '233604007', '67782005', '84114007', '22298006']  # Sepsis, Pneumonia, ARDS, Heart Failure, MI
valid_snomed_count = 0
for code in test_snomed:
    if ontology_validator.validate_snomed_code(code):
        valid_snomed_count += 1

print(f"✓ SNOMED-CT validation: {valid_snomed_count}/{len(test_snomed)} test codes valid")

print("✓ Complete ontology validation completed")

Initializing ontology validator...
2026-03-26 11:09:40,838 - utils.ontology_validator - INFO - Initialized OntologyValidator
2026-03-26 11:09:40,839 - utils.ontology_validator - INFO -   LOINC codes: 61
2026-03-26 11:09:40,839 - utils.ontology_validator - INFO -   SNOMED codes: 50
2026-03-26 11:09:40,840 - utils.ontology_validator - INFO -   ICD-10 codes: 52
2026-03-26 11:09:40,840 - utils.ontology_validator - INFO -   Reference ranges: 32
Validating ICD-10 codes...
✓ ICD-10 validation: 2/100 sample codes valid
  Total ICD diagnoses available: 6364488
Validating LOINC codes...
✓ Loaded 61 LOINC codes in ontology
✓ LOINC validation: 5/5 test codes valid
Validating SNOMED-CT codes...
✓ Loaded 50 SNOMED-CT codes in ontology
✓ SNOMED-CT validation: 5/5 test codes valid
✓ Complete ontology validation completed


In [9]:
# Import SFT training components
from training.sft_trainer import run_sft_training
from models.mdt_model import MedicalDigitalTwinModel
from config.configs import ModelConfig, SFTConfig
from data.dataset import CognitiveStreamDataset

# Initialize model configuration
model_config = ModelConfig()
print(f"✓ Model config loaded: {model_config.model_name}")

# Initialize SFT training configuration
sft_config = SFTConfig()
print(f"✓ SFT config loaded: {sft_config.num_epochs} epochs")

# Initialize model
model = MedicalDigitalTwinModel(model_config)

# Prepare training data split
if len(all_training_data) > 0:
    # Split data for training and evaluation (90% train, 10% eval)
    split_idx = int(len(all_training_data) * 0.9)
    train_data = all_training_data[:split_idx]
    eval_data = all_training_data[split_idx:]
    
    # Create dataset objects
    train_dataset = CognitiveStreamDataset(
        train_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    eval_dataset = CognitiveStreamDataset(
        eval_data,
        model.tokenizer,
        max_length=model_config.max_length
    )
    
    print(f"✓ Data split: {len(train_dataset)} train, {len(eval_dataset)} eval examples")
else:
    train_dataset = None
    eval_dataset = None
    print("⚠️ No training data available")

# Run SFT training
print("Starting SFT training...")
if train_dataset is not None and eval_dataset is not None:
    trained_model = run_sft_training(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        config=sft_config
    )
    print("✓ SFT training completed")
else:
    print("⚠️ Skipping SFT training - no data available")
    trained_model = model

✓ Model config loaded: Qwen/Qwen3.5-4B
✓ SFT config loaded: 3 epochs
2026-03-26 11:09:45,309 - models.mdt_model - WARNING - Using GPT-2 for demo. Set use_demo_model=False for production.
2026-03-26 11:09:45,323 - models.mdt_model - INFO - Loading GPT-2 demo model...
2026-03-26 11:09:45,418 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 11:09:45,461 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 11:09:45,506 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-03-26 11:09:45,507 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

2026-03-26 11:09:45,702 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-03-26 11:09:45,749 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 11:09:45,796 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:09:45,842 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-26 11:09:45,887 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:09:45,934 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expan

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


...

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-26 11:11:53,591 - training.sft_trainer - INFO - 
2026-03-26 11:11:53,592 - training.sft_trainer - INFO - Saving final model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-26 11:11:54,349 - training.sft_trainer - INFO - 
2026-03-26 11:11:54,349 - training.sft_trainer - INFO - ================================================================================
2026-03-26 11:11:54,350 - training.sft_trainer - INFO - SUPERVISED FINE-TUNING COMPLETE
2026-03-26 11:11:54,350 - training.sft_trainer - INFO - ================================================================================
2026-03-26 11:11:54,350 - training.sft_trainer - INFO -   Final loss: 1.2715
2026-03-26 11:11:54,351 - training.sft_trainer - INFO -   Total steps: 438
2026-03-26 11:11:54,351 - training.sft_trainer - INFO -   Model saved to: outputs/sft/final_model
2026-03-26 11:11:54,351 - training.sft_trainer - INFO -   Logs saved to: outputs/logs/sft
2026-03-26 11:11:54,352 - training.sft_trainer - INFO - ================================================================================
2026-03-26 11:11:54,352 - training.sft_trainer - INFO - 
✓ SFT training completed


In [ ]:
# View SFT Training Results
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

print("--- SFT Training Results ---")

metrics_path = "outputs/sft/training_metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
        
    print(f"Final Training Loss: {metrics.get('train_loss', 'N/A'):.4f}")
    print(f"Total Steps: {metrics.get('total_steps', 'N/A')}")
    print(f"Total Epochs: {metrics.get('epoch', 'N/A')}")
    print(f"Runtime: {metrics.get('train_runtime', 0):.2f} seconds")
    print(f"Samples per second: {metrics.get('train_samples_per_second', 'N/A')}")
else:
    print("Could not find Hugging Face Trainer metrics. Checking for basic loop history...")

history_path = "outputs/sft/training_history.json"
if os.path.exists(history_path):
    with open(history_path, "r") as f:
        history = json.load(f)
    if 'training_losses' in history and len(history['training_losses']) > 0:
        plt.figure(figsize=(8, 4))
        plt.plot(history['training_losses'], marker='o', label='Training Loss')
        plt.title('SFT Training Curve')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.legend()
        plt.show()

# If tensorboard logs exist, we can enable it inline
print("\nTo view detailed interactive loss curves, you can run this command in a new cell:")
print("%load_ext tensorboard")
print("%tensorboard --logdir outputs/sft/runs/")

## 5. Alignment Training (GRPO)

In [ ]:
# Import GRPO training components
from training.grpo_trainer import run_grpo_training
from config.configs import GRPOConfig
from data.dataset import CognitiveStreamDataset
from torch.utils.data import DataLoader

# Initialize reward engine
reward_engine = CompositeRewardEngine()

# Create GRPO configuration
grpo_config = GRPOConfig()
print(f"✓ GRPO config loaded: {grpo_config.num_iterations} iterations")

# Create dataloader for GRPO training
if train_dataset is not None:
    # Use batch_size=1 for GRPO since it processes prompts individually
    grpo_dataloader = DataLoader(
        train_dataset,
        batch_size=1,  # Process one prompt at a time for GRPO
        shuffle=True
    )
    print(f"✓ Created GRPO dataloader with {len(train_dataset)} examples")
else:
    grpo_dataloader = None
    print("⚠️ No training data available for GRPO")

# Run GRPO alignment
print("Starting GRPO alignment training...")
if grpo_dataloader is not None:
    aligned_model = run_grpo_training(
        model=trained_model,
        train_dataloader=grpo_dataloader,
        config=grpo_config,
        reward_engine=reward_engine
    )
    print("✓ GRPO alignment completed")
else:
    print("⚠️ Skipping GRPO training - no dataloader available")
    aligned_model = trained_model

2026-03-26 11:29:53,461 - absl - INFO - Using default tokenizer.
2026-03-26 11:29:53,462 - rewards.semantic_reward - INFO - Initialized SemanticFidelityReward
2026-03-26 11:29:53,473 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
2026-03-26 11:29:53,566 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:29:53,578 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
2026-03-26 11:29:53,620 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:29:53,631 - httpx - INFO - HTTP Request: HEAD https://huggingface.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-03-26 11:29:54,322 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:29:54,333 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config.json "HTTP/1.1 200 OK"
2026-03-26 11:29:54,376 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 11:29:54,388 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 11:29:54,435 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-mpnet-base-v2/tree/main/additional_chat_templates?recursive

Prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-03-26 13:02:19,377 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-26 13:02:19,424 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 13:02:19,470 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 13:02:19,514 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-26 13:02:19,558 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-26 13:02:19,606 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator
from core.cognitive_streams import CognitiveStreamParser

# Check if reward_engine is initialized earlier in the notebook, if not create one
if 'reward_engine' not in locals():
    print("Warning: reward_engine not found, initializing a new one for evaluation.")
    from rewards.composite_engine import CompositeRewardEngine
    reward_engine = CompositeRewardEngine()

# Initialize parser
parser = CognitiveStreamParser()

print("Initializing comprehensive evaluation framework...")
# Run evaluation
evaluator = MedicalTwinEvaluator(
    model=aligned_model,
    parser=parser,
    reward_engine=reward_engine
)

# Run the 8 predefined benchmark clinical cases
results_df = evaluator.run_evaluation()

# Generate visual report
report = evaluator.generate_report(results_df)

print("✓ Evaluation completed")
print(report)

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator

# Run evaluation
evaluator = MedicalTwinEvaluator()
results = evaluator.evaluate_model(aligned_model)

print("✓ Evaluation completed")
print(f"Results: {results}")


: 

: 

: 

## 7. Saving Results & Interactive Testing (Gradio)
Save the final aligned model weights, dump the evaluation metrics, and launch a simple Gradio web interface to manually test the Medical Digital Twin with hypothetical patient cases.

In [ ]:
import json
import os
import torch
from datetime import datetime

# Optional: Ensure gradio is installed
try:
    import gradio as gr
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio"])
    import gradio as gr

# 1. Save Training & Evaluation Results
print("Saving results and model...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("results", exist_ok=True)
results_path = f"results/final_eval_results_{timestamp}.json"
report_path = f"results/evaluation_report_{timestamp}.txt"

# Save the detailed pandas dataframe as a CSV
if 'results_df' in locals():
    csv_path = f"results/evaluation_results_{timestamp}.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"✓ Full evaluation dataframe saved to {csv_path}")

# Save the formatted text report
if 'report' in locals():
    with open(report_path, "w") as f:
        f.write(report)
    print(f"✓ Summary report saved to {report_path}")

# 2. Save Trained Model
model_save_path = f"outputs/final_model_{timestamp}"
os.makedirs(model_save_path, exist_ok=True)
try:
    if hasattr(aligned_model, 'model') and hasattr(aligned_model, 'tokenizer'):
        aligned_model.model.save_pretrained(model_save_path)
        aligned_model.tokenizer.save_pretrained(model_save_path)
        print(f"✓ Model successfully saved to {model_save_path}")
    elif hasattr(aligned_model, 'save_pretrained'): # If it's a raw transformers model
        aligned_model.save_pretrained(model_save_path)
        print(f"✓ Model successfully saved to {model_save_path}")
except Exception as e:
    print(f"⚠️ Could not save model automatically: {e}")

# 3. Gradio Interface for Interactive Testing
def predict_medical_case(patient_history, current_symptoms):
    if not hasattr(aligned_model, 'model') or not hasattr(aligned_model, 'tokenizer'):
        return "Model not properly loaded! Please ensure training finished."
        
    prompt = f"Patient History: {patient_history}\n\nCurrent Symptoms: {current_symptoms}\n\nProvide a clinical assessment and recommendations:"
    
    try:
        # Generate the response using our custom evaluation generate function if possible
        if hasattr(aligned_model, 'generate'):
            raw_response = aligned_model.generate(prompt, max_length=1024)
            # Check if parser is available to extract structured streams
            if 'parser' in locals():
                streams = parser.parse(raw_response)
                formatted_response = (
                    f"🤔 **Theory of Mind (Internal)**:\n{streams.think or 'N/A'}\n\n"
                    f"📋 **Patient State**:\n{streams.patient_state or 'N/A'}\n\n"
                    f"🩺 **Final Physician Response**:\n{streams.response or raw_response}"
                )
                return formatted_response
            return raw_response
        else:
            return "aligned_model missing generate method."
            
    except Exception as e:
        return f"Error during generation: {str(e)}"

print("\n🚀 Launching Gradio Interface...")
demo = gr.Interface(
    fn=predict_medical_case,
    inputs=[
        gr.Textbox(lines=4, label="Patient History", placeholder="e.g., 45yo male with history of hypertension..."),
        gr.Textbox(lines=3, label="Current Symptoms", placeholder="e.g., severe chest pain radiating to left arm...")
    ],
    outputs=gr.Markdown(label="MDT Assessment & Recommendations"),
    title="🩺 Metacognitive Medical Digital Twin",
    description="Interactive testing interface for the trained MDT model. Simulates reasoning ('Theory of Mind') before returning an empathetic, safe physician response."
)

# Launch Gradio inline directly below
demo.launch(share=True, inline=True)